In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import torch
import torch_geometric as pyg
import mrmr

from baseline_evals.feature_selection import variance_filtering

from torch_geometric.utils import to_networkx


from bipartite_gnn.train_test import GNNTrainer
from bipartite_gnn.model import GAT_2L, BipartiteRGAT, BiRGAT
from bipartite_gnn.graph_building import cosine_similarity_matrix, dense_to_coo
from bipartite_gnn.preprocessing import gg_interactions, get_mirna_gene_interactions, pp_interactions, gg_interactions
from bipartite_gnn.graph_building import create_diff_exp_connections_nbnom, create_diff_exp_connections_norm, dense_to_coo
from baseline_evals.feature_selection import class_variational_selection, variance_filtering

In [3]:
# Check currently allocated CUDA memory in bytes
allocated_memory = torch.cuda.memory_allocated()
print(f"Currently allocated CUDA memory: {allocated_memory} bytes")

# Check maximum allocated CUDA memory in bytes
max_allocated_memory = torch.cuda.max_memory_allocated()
print(f"Maximum allocated CUDA memory: {max_allocated_memory} bytes")


Currently allocated CUDA memory: 0 bytes
Maximum allocated CUDA memory: 0 bytes


In [4]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnv.tsv", separator="\t", null_values=null_vals)
cna_th = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())
# labels, y

In [5]:
cna

Gene Symbol,TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SH-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SO-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CP-01,TCGA-A2-A0CQ-01,TCGA-A2-A0CS-01,TCGA-A2-A0CT-01,TCGA-A2-A0CU-01,TCGA-A2-A0CV-01,TCGA-A2-A0CW-01,TCGA-A2-A0D0-01,TCGA-A2-A0D1-01,TCGA-A2-A0D2-01,TCGA-A2-A0D3-01,TCGA-A2-A0D4-01,TCGA-A2-A0EM-01,TCGA-A2-A0EN-01,TCGA-A2-A0EO-01,TCGA-A2-A0EQ-01,TCGA-A2-A0ER-01,…,TCGA-E2-A14W-01,TCGA-E2-A14X-01,TCGA-E2-A14Y-01,TCGA-E2-A14Z-01,TCGA-E2-A150-01,TCGA-E2-A152-01,TCGA-E2-A153-01,TCGA-E2-A154-01,TCGA-E2-A155-01,TCGA-E2-A156-01,TCGA-E2-A158-01,TCGA-E2-A159-01,TCGA-E2-A15A-01,TCGA-E2-A15C-01,TCGA-E2-A15D-01,TCGA-E2-A15E-01,TCGA-E2-A15F-01,TCGA-E2-A15G-01,TCGA-E2-A15H-01,TCGA-E2-A15I-01,TCGA-E2-A15J-01,TCGA-E2-A15K-01,TCGA-E2-A15L-01,TCGA-E2-A15M-01,TCGA-E2-A15O-01,TCGA-E2-A15P-01,TCGA-E2-A15R-01,TCGA-E2-A15S-01,TCGA-E2-A15T-01,TCGA-E2-A1AZ-01,TCGA-E2-A1B0-01,TCGA-E2-A1B1-01,TCGA-E2-A1B4-01,TCGA-E2-A1B5-01,TCGA-E2-A1B6-01,TCGA-E2-A1BC-01,TCGA-E2-A1BD-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AJAP1""",-0.521,-0.06,-0.398,-0.022,0.475,-0.595,-0.502,0.035,0.18,-0.04,-0.852,-0.085,-0.09,0.055,0.128,-0.101,0.462,-0.03,0.123,0.013,-0.008,-0.558,-0.412,0.027,-0.01,-0.66,0.28,0.02,-0.085,0.0,-0.315,-0.01,-0.352,-0.289,-0.038,-0.008,…,0.094,0.323,0.023,0.47,0.338,-0.081,-0.081,-0.526,0.012,0.0,-0.907,0.287,-0.489,0.041,0.006,-0.081,-0.133,-0.031,-0.656,0.06,-0.001,-0.023,0.048,-0.009,-0.771,-0.008,-0.844,0.1,0.045,0.244,0.057,-0.001,0.005,0.031,-0.011,-0.006,0.005
"""CAMTA1""",-0.521,-0.06,-0.412,-0.022,0.013,-0.595,-0.502,0.035,-0.576,-0.04,-0.852,-0.085,-0.175,0.055,1.369,-0.101,0.462,-0.03,0.123,0.013,-0.008,-0.558,-0.412,0.027,-0.01,-0.66,0.28,0.02,-0.085,0.0,-0.315,-0.01,-0.352,-0.289,-0.038,-0.008,…,0.094,0.323,0.023,0.47,0.338,-0.081,-0.081,-0.526,0.012,0.0,-0.907,0.287,-0.489,0.041,0.006,-0.081,-0.133,-0.031,-0.656,0.06,-0.001,-0.023,0.048,-0.009,-0.771,-0.008,-0.844,-0.771,0.045,0.244,0.057,-0.001,0.005,0.031,-0.011,-0.006,0.005
"""RN7SL729P""",-0.521,-0.06,-0.412,-0.022,0.013,-0.595,-0.561,0.035,0.454,-0.04,-0.852,-0.085,-0.175,0.055,0.301,-0.101,0.462,-0.03,0.123,0.013,-0.008,-0.558,-0.412,0.027,-0.01,-0.66,0.28,0.02,-0.085,0.0,-0.315,-0.01,-0.352,-0.289,-0.038,-0.008,…,0.094,0.323,0.023,0.47,0.338,-0.081,-0.081,-0.526,0.012,0.0,-0.907,0.287,-0.489,0.041,0.006,-0.081,-0.133,-0.031,-0.656,0.06,-0.001,-0.023,0.014,-0.009,-0.771,-0.008,-0.844,-0.771,0.045,0.244,0.057,-0.001,0.005,0.031,-0.011,-0.006,0.005
"""SLC45A1""",-0.521,-0.06,-0.412,-0.022,0.013,-0.595,-0.561,0.035,0.454,-0.04,-0.852,-0.085,-0.175,0.055,0.301,-0.101,0.462,-0.03,0.123,0.013,-0.008,-0.558,-0.412,0.027,-0.01,-0.66,0.28,0.02,-0.085,0.0,-0.315,-0.01,-0.352,-0.289,-0.038,-0.008,…,0.094,0.323,0.023,0.47,0.338,-0.081,-0.081,-0.526,0.012,0.0,-0.907,0.287,-0.489,0.041,0.006,-0.081,-0.133,-0.031,-0.656,0.06,-0.001,-0.023,0.014,-0.009,-0.771,-0.008,-0.844,-0.771,0.045,0.244,0.057,-0.001,0.005,0.031,-0.011,-0.006,0.005
"""RERE""",-0.521,-0.06,-0.412,-0.022,0.013,-0.595,-0.561,0.035,0.454,-0.04,-0.852,-0.085,-0.175,0.055,0.301,-0.101,0.462,-0.03,0.123,0.013,-0.008,-0.558,-0.412,0.027,-0.01,-0.66,0.28,0.02,-0.085,0.0,-0.315,-0.01,-0.352,-0.289,-0.038,-0.008,…,0.094,0.323,0.023,0.47,0.338,-0.081,-0.081,-0.526,0.012,0.0,-1.293,0.287,-0.489,0.041,0.006,-0.081,-0.133,-0.031,-0.656,0.06,-0.001,-0.023,0.014,-0.009,-0.771,-0.008,-0.844,-0.771,0.045,0.244,0.057,-0.001,0.005,0.031,-0.011,-0.006,0.005
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…

In [6]:
cna_th = cna_th.filter(
    pl.col("Gene Symbol").is_in(cna["Gene Symbol"])
)

In [7]:
assert mrna.columns[1:] == cna.columns[1:] == cna_th.columns[1:] == mirna.columns[1:], "Columns do not match"

In [8]:
# mrna, cna, mirna
mrna_gene_names = mrna[:, 0].to_list()
cna_gene_names = cna[:, 0].to_list()
mirna_gene_names = mirna[:, 0].to_list()

In [9]:
mirna_gene_names = pl.read_csv("miRBaseConverter_Result.csv")
mirna_targets = mirna_gene_names["TargetName"].to_list()

In [10]:
mrna_X = torch.tensor(mrna[:, 1:].to_numpy().T)
cna_X = torch.tensor(cna[:, 1:].to_numpy().T)
mirna_X = torch.tensor(mirna[:, 1:].to_numpy().T)

random_state = 44

train_mask, val_test_mask = train_test_split(torch.arange(len(y)), test_size=0.2, stratify=y, random_state=random_state)
val_mask, test_mask = train_test_split(val_test_mask, test_size=0.5, random_state=random_state, stratify=y[val_test_mask])

In [11]:
np.unique(y[train_mask], return_counts=True), np.unique(y[val_mask], return_counts=True), np.unique(y[test_mask], return_counts=True)

((array([0, 1, 2, 3]), array([ 69,  44, 176,  97])),
 (array([0, 1, 2, 3]), array([ 9,  5, 22, 12])),
 (array([0, 1, 2, 3]), array([ 9,  6, 22, 12])))

In [12]:
from bipartite_gnn.preprocessing import mrmr_selection

mrna_X_selected = mrmr_selection(
    X_df=mrna, 
    y=y, 
    train_mask=train_mask,
    n_features=100,
    n_preselected_features=2000,
    feature_col_name="sample"
)

cnv_X_selected = mrmr_selection(
    X_df=cna, 
    y=y, 
    train_mask=train_mask,
    n_features=100,
    n_preselected_features=2000,
    feature_col_name="Gene Symbol"
)

mirna_X_selected = mrmr_selection(
    X_df=mirna, 
    y=y, 
    train_mask=train_mask,
    n_features=100,
    n_preselected_features=None,
    feature_col_name="sample"
)
# mrna_X_selected

100%|██████████| 100/100 [00:02<00:00, 48.45it/s]


In [13]:
mrna_X = torch.tensor(mrna_X_selected[:, 1:].to_numpy().T)
mrna_genes = mrna_X_selected[:, 0].to_list()

cna_X = torch.tensor(cnv_X_selected[:, 1:].to_numpy().T)
cna_genes = cnv_X_selected[:, 0].to_list()

mirna_X = torch.tensor(mirna_X_selected[:, 1:].to_numpy().T)
mirna_genes = mirna_X_selected[:, 0].to_list()

In [14]:
mirna_A = create_diff_exp_connections_norm(mirna_X, train_mask, 1.0)

isolated sample nodes, isolated gene nodes, mean degree: 
tensor(0) tensor(0) tensor(30.2381, dtype=torch.float64)


In [15]:
# construct the A matrix for cna
cna_th = cna_th.filter(pl.col("Gene Symbol").is_in(cna_genes))

cna_A = cna_th[:,1:].to_numpy().T.astype(np.float32)
cna_A = torch.tensor(cna_A, dtype=torch.float32)

print((cna_A.sum(dim=1) == 0).sum(), (cna_A.sum(dim=0) == 0).sum())

tensor(15) tensor(0)


In [16]:
mrna_A = create_diff_exp_connections_norm(mrna_X, train_mask, 1.0)
mrna_A_n = create_diff_exp_connections_nbnom(mrna_X, train_mask, 1.0)
mrna_A = torch.logical_or(mrna_A, mrna_A_n).int()

print((mrna_A.abs().sum(dim=1) == 0).sum(), (mrna_A.abs().sum(dim=0) == 0).sum())

isolated sample nodes, isolated gene nodes, mean degree: 
tensor(0) tensor(0) tensor(28.2091, dtype=torch.float64)
isolated sample nodes, isolated gene nodes, mean degree: 
tensor(307) tensor(80) tensor(1.1366, dtype=torch.float64)
tensor(0) tensor(0)


In [17]:
# connected_nodes_mask = mrna_A.sum(dim=0) != 0
# # connected_nodes_mask_cna = cna_A.sum(dim=0) != 0

# mrna_A = mrna_A[:, connected_nodes_mask]
# mrna_X = mrna_X[:, connected_nodes_mask]

# # cna_A = cna_A[:, connected_nodes_mask_cna]
# # cna_X = cna_X[:, connected_nodes_mask_cna]

# # mrna_genes = mrna_genes[connected_nodes_mask]
# # cna_genes = cna_genes[connected_nodes_mask_cna]

# print(mrna_A.shape, mrna_X.shape)

# normalize
std_scaler = StandardScaler()
std_scaler.fit(mrna_X.numpy()[train_mask])
mrna_X = torch.tensor(std_scaler.transform(mrna_X.numpy()))

std_scaler = StandardScaler()
std_scaler.fit(cna_X.numpy()[train_mask])
cna_X = torch.tensor(std_scaler.transform(cna_X.numpy()))

std_scaler = StandardScaler()
std_scaler.fit(mirna_X.numpy()[train_mask])
mirna_X = torch.tensor(std_scaler.transform(mirna_X.numpy()))


In [18]:
gg_A = gg_interactions(mrna_genes, mrna_genes)
pp_A = pp_interactions(mrna_genes, mrna_genes)

mrna_features_A = torch.logical_or(gg_A, pp_A).int()

gg_A = gg_interactions(cna_genes, cna_genes)
pp_A = pp_interactions(cna_genes, cna_genes)

cna_features_A = torch.logical_or(gg_A, pp_A).int()

mrna_features_A.sum(), cna_features_A.sum()

(tensor(135), tensor(69))

In [19]:
# write mirna_genes to a csv file
mirna_genes_df = pl.DataFrame(mirna_genes)
mirna_genes_df.write_csv("mirna_genes.csv")

### Use mirbase converter to convert ascession names (MIMAT) to miRNA name

In [20]:
mirna_genes_names = pl.read_csv('miRBaseConverter_Result.csv')
mirna_genes = mirna_genes_names['TargetName'].to_list()[1:]

In [21]:
mirna_mrna_interactions = get_mirna_gene_interactions(mirna_genes, mrna_genes)
mirna_cna_interactions = get_mirna_gene_interactions(mirna_genes, cna_genes)

mc_A_gg = gg_interactions(mrna_genes, cna_genes)
mc_A_pp = pp_interactions(mrna_genes, cna_genes)

mc_A = torch.logical_or(mc_A_gg, mc_A_pp).int()

torch.Size([50, 100])
torch.Size([50, 100])


In [22]:
mrna_genes.__len__(), cna_genes.__len__(), mirna_genes.__len__()

(100, 100, 50)

In [23]:
import torch_geometric.transforms as T

from bipartite_gnn.graph_building import dense_to_attributes

data = pyg.data.HeteroData()

proj_dim = mrna_X.shape[1]

# sample node features
data["mrna"].x = mrna_X.float()
data["cna"].x = cna_X.float()
data["mirna"].x = mirna_X.float()

data.y = torch.tensor(y)

data.omics = ["mrna", "cna", "mirna"]
data.feature_names = ["mrna_feature", "cna_feature", "mirna_feature"]

# feature node projection
data["mrna_feature"].x = torch.ones(mrna_X.shape[1], proj_dim)
data["cna_feature"].x = torch.ones(cna_X.shape[1], proj_dim)
data["mirna_feature"].x = torch.ones(mirna_X.shape[1], proj_dim)

# create edges
data["mrna", "diff_exp", "mrna_feature"].edge_index = dense_to_coo(mrna_A)
data["mrna", "diff_exp", "mrna_feature"].edge_attributes = dense_to_attributes(mrna_A)
data["mrna_feature", "interaction", "mrna_feature"].edge_index = dense_to_coo(mrna_features_A)

data["cna", "diff_exp", "cna_feature"].edge_index = dense_to_coo(cna_A)
data["cna", "diff_exp", "cna_feature"].edge_attributes = dense_to_attributes(cna_A)
data["cna_feature", "interaction", "cna_feature"].edge_index = dense_to_coo(cna_features_A)

# MIRNA INTERACTIONS
data["mirna", "diff_exp", "mirna_feature"].edge_index = dense_to_coo(mirna_A)
data["mirna", "diff_exp", "mirna_feature"].edge_attributes = dense_to_attributes(mirna_A)

data["mirna_feature", "interacts", "mrna_feature"].edge_index = dense_to_coo(mirna_mrna_interactions)
data["mirna_feature", "interacts", "cna_feature"].edge_attributes = dense_to_coo(mirna_cna_interactions)

data["mrna_feature", "interacts", "cna_feature"].edge_index = dense_to_coo(mc_A)

data = T.ToUndirected()(data)
# data = T.AddSelfLoops()(data)

# 2 for forward and backward diff exp, one for each self loop
data.num_relations = len(data.edge_index_dict.keys())
print("num_relations", data.num_relations)

data["train_mask"] = train_mask
data["val_mask"] = val_mask
data["test_mask"] = test_mask

data

num_relations 12


HeteroData(
  y=[483],
  omics=[3],
  feature_names=[3],
  num_relations=12,
  train_mask=[386],
  val_mask=[48],
  test_mask=[49],
  mrna={ x=[483, 100] },
  cna={ x=[483, 100] },
  mirna={ x=[483, 100] },
  mrna_feature={ x=[100, 100] },
  cna_feature={ x=[100, 100] },
  mirna_feature={ x=[100, 100] },
  (mrna, diff_exp, mrna_feature)={
    edge_index=[2, 13641],
    edge_attributes=[13641, 1],
  },
  (mrna_feature, interaction, mrna_feature)={ edge_index=[2, 173] },
  (cna, diff_exp, cna_feature)={
    edge_index=[2, 24316],
    edge_attributes=[24316, 1],
  },
  (cna_feature, interaction, cna_feature)={ edge_index=[2, 82] },
  (mirna, diff_exp, mirna_feature)={
    edge_index=[2, 14605],
    edge_attributes=[14605, 1],
  },
  (mirna_feature, interacts, mrna_feature)={ edge_index=[2, 74] },
  (mirna_feature, interacts, cna_feature)={ edge_attributes=[2, 90] },
  (mrna_feature, interacts, cna_feature)={ edge_index=[2, 67] },
  (mrna_feature, rev_diff_exp, mrna)={
    edge_index=[2, 1

In [30]:
model = BiRGAT(
    omic_channels=data.omics,
    feature_names=data.feature_names,
    relations=list(data.edge_index_dict.keys()),
    input_dims=[mrna_X.shape[1], cna_X.shape[1], mirna_X.shape[1]],
    num_classes=len(np.unique(y)),
    proj_dim=proj_dim,
    hidden_channels=[128, 128, 128, 16],
    heads=4,
    dropout=0.4,
)

# model = GAT_2L(
#     input_shape=mrna_X.shape[1], # , cna_X_400.shape[1], mirna_X_200.shape[1]],
#     num_labels=len(np.unique(y)),
#     proj_dim=proj_dim,
#     hidden_channels=512, # num_layers = len(hidden_channels) - 1
#     num_heads=4,
#     dropout=0.6,
#     eps=0.99,
# )

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)

ccounts = torch.unique(data.y[train_mask], return_counts=True)[1]
inv_w = 1.0 / ccounts.float()
class_weights = inv_w / inv_w.sum()

trainer = GNNTrainer(
    model=model,
    loss_fn=torch.nn.CrossEntropyLoss(weight=class_weights),
    optimizer=optimizer,
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=15, min_lr=1e-5),
    params={
        "l1_lambda" : 0.0,
    }
)

trainer.train(
    data=data,
    epochs=250,
    log_interval=5,
)

Epoch: 005, 
Train Loss: 1.3645, Train Acc: 0.2979, Train F1 Macro: 0.2046, Train F1 Weighted: 0.2891
Val Loss: 1.3594, Val Acc: 0.3750, Val F1 Macro: 0.2942, Val F1 Weighted: 0.3906
Test Loss: 1.3596, Test Acc: 0.3061, Test F1 Macro: 0.2342, Test F1 Weighted: 0.3021
##################################################
Epoch: 010, 
Train Loss: 1.2888, Train Acc: 0.7979, Train F1 Macro: 0.7389, Train F1 Weighted: 0.7887
Val Loss: 1.2686, Val Acc: 0.7083, Val F1 Macro: 0.5622, Val F1 Weighted: 0.6681
Test Loss: 1.2873, Test Acc: 0.7755, Test F1 Macro: 0.6226, Test F1 Weighted: 0.7297
##################################################
Epoch: 015, 
Train Loss: 1.2227, Train Acc: 0.4404, Train F1 Macro: 0.3535, Train F1 Weighted: 0.4401
Val Loss: 1.2941, Val Acc: 0.2708, Val F1 Macro: 0.1611, Val F1 Weighted: 0.1319
Test Loss: 1.3297, Test Acc: 0.2449, Test F1 Macro: 0.1091, Test F1 Weighted: 0.1069
##################################################
Epoch: 020, 
Train Loss: 1.1323, Train Acc:

In [25]:
from bipartite_gnn.feat_integration_models import construct_cross_feature_discovery_tensor, construct_cross_feature_discovery_tensor_single

a = torch.tensor([[1, 2, 3, 5], [4, 5, 6, 8]])
b = a.repeat(2, 1, 1)
a, b

(tensor([[1, 2, 3, 5],
         [4, 5, 6, 8]]),
 tensor([[[1, 2, 3, 5],
          [4, 5, 6, 8]],
 
         [[1, 2, 3, 5],
          [4, 5, 6, 8]]]))

In [26]:
construct_cross_feature_discovery_tensor(b)

tensor([[[1, 2, 3, 5],
         [1, 2, 3, 5]],

        [[4, 5, 6, 8],
         [4, 5, 6, 8]]])


tensor([[ 1.,  2.,  3.,  5.,  2.,  4.,  6., 10.,  3.,  6.,  9., 15.,  5., 10.,
         15., 25.],
        [16., 20., 24., 32., 20., 25., 30., 40., 24., 30., 36., 48., 32., 40.,
         48., 64.]])

In [27]:
construct_cross_feature_discovery_tensor_single(a)

tensor([ 4,  5,  6,  8,  8, 10, 12, 16, 12, 15, 18, 24, 20, 25, 30, 40])

In [28]:
torch.outer(a[0], a[1])

tensor([[ 4,  5,  6,  8],
        [ 8, 10, 12, 16],
        [12, 15, 18, 24],
        [20, 25, 30, 40]])